In [1]:
# Import numpy for numerical operations and array manipulations
import numpy as np
# Import pandas for tabular data manipulation, filtering, and merging DataFrames
import pandas as pd
# Import ast (Abstract Syntax Tree) to safely evaluate string-formatted lists/dictionaries from CSV columns into actual Python objects
import ast

In [2]:
# Load the movies metadata dataset containing budget, genres, overview, popularity, etc.
movies = pd.read_csv('tmdb_5000_movies.csv')
# Load the credits dataset containing movie_id, title, cast, and crew members
credits = pd.read_csv('tmdb_5000_credits.csv')

In [3]:
# Display the first 2 rows of the movies DataFrame to inspect columns, data formats, and contents
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [4]:
# Display the first 2 rows of the credits DataFrame to check cast and crew structure
credits.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [5]:
# Merge the movies and credits DataFrames on the common 'title' column so all information is in one single DataFrame
movies = movies.merge(credits, on='title')
# Check the total number of rows and columns in the merged DataFrame
movies.shape

(4809, 23)

In [6]:
# Select only the essential features that help understand movie content for similarity calculation
# movie_id: unique identifier for posters/web-apps
# title: name of the movie
# overview: summary of the plot
# genres: movie category (Action, Comedy, etc.)
# keywords: tags describing specific themes
# cast: actors in the movie
# crew: directors and production crew
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
# View the top 5 rows of the filtered DataFrame
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [7]:
# Check the count of missing (null) values in each column to identify incomplete data
print(movies.isnull().sum())
# Drop rows with missing values (e.g., missing overview) to avoid runtime errors during text processing
movies.dropna(inplace=True)
# Count duplicate movie rows to verify dataset uniqueness
print(f"Duplicates: {movies.duplicated().sum()}")

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64
Duplicates: 0


In [8]:
# Define a helper function to extract name values from JSON string representations of lists of dictionaries
def convert(obj):
    # Initialize an empty list to store extracted names
    L = []
    # Parse string into python list using ast.literal_eval and loop through each dictionary
    for i in ast.literal_eval(obj):
        # Append the 'name' attribute value to list L
        L.append(i['name'])
    # Return list of names (e.g., ['Action', 'Adventure'])
    return L

# Define a helper function to extract only the top 3 lead cast members (actors)
def convert3(obj):
    # Initialize an empty list for top 3 actors
    L = []
    # Counter to limit the selection to at most 3 actors
    counter = 0
    # Parse and iterate over the cast list
    for i in ast.literal_eval(obj):
        # Take actor name if counter is less than 3
        if counter < 3:
            L.append(i['name'])
            # Increment counter
            counter += 1
        else:
            # Break out of loop once 3 actors are gathered
            break
    # Return the list of top 3 actors
    return L

# Define a helper function to extract the Director from the crew column
def fetch_director(obj):
    # Initialize an empty list for director name
    L = []
    # Parse and iterate over all crew members
    for i in ast.literal_eval(obj):
        # Check if the crew member's job is 'Director'
        if i['job'] == 'Director':
            # Append director's name
            L.append(i['name'])
            # Break because usually the main director is what we need
            break
    # Return list containing director name
    return L

In [9]:
# Apply convert function on 'genres' column to turn stringified JSON into clean Python list of genre names
movies['genres'] = movies['genres'].apply(convert)
# Apply convert function on 'keywords' column to extract list of keywords
movies['keywords'] = movies['keywords'].apply(convert)
# Apply convert3 function on 'cast' column to extract top 3 actor names
movies['cast'] = movies['cast'].apply(convert3)
# Apply fetch_director function on 'crew' column to extract the director's name
movies['crew'] = movies['crew'].apply(fetch_director)
# Split overview string into a list of individual words so it can be combined with other word lists
movies['overview'] = movies['overview'].apply(lambda x: x.split())
# Preview the transformed DataFrame with list-structured columns
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [10]:
# Remove spaces inside genres to prevent confusion (e.g., 'Science Fiction' -> 'ScienceFiction')
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
# Remove spaces inside keywords so multi-word keywords are treated as unique single tokens
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
# Remove spaces in cast names so 'Sam Worthington' becomes 'SamWorthington' (prevents mixing up with another 'Sam')
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
# Remove spaces in director names (e.g., 'James Cameron' -> 'JamesCameron')
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

# Concatenate overview words, genres, keywords, cast, and director lists into one comprehensive 'tags' list
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
# Create a new compact DataFrame with only the essential columns needed for recommendation
new_df = movies[['movie_id', 'title', 'tags']]
# Join the list of tag words into a single continuous lower-case text string
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())
# Display the top 5 records of new_df
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [11]:
# Import PorterStemmer from NLTK to reduce words to their root/base form (e.g., 'dancing', 'danced' -> 'danc')
from nltk.stem.porter import PorterStemmer
# Instantiate the PorterStemmer object
ps = PorterStemmer()

# Define a stemming function to process each string of tags
def stem(text):
    # Initialize a list to hold stemmed words
    y = []
    # Loop through each word in the text
    for i in text.split():
        # Stem word and append to list
        y.append(ps.stem(i))
    # Rejoin words back into a single string separated by space
    return " ".join(y)

# Apply the stem function to all movie tag strings in new_df
new_df['tags'] = new_df['tags'].apply(stem)

In [12]:
# Import CountVectorizer from scikit-learn to convert text tags into bag-of-words numerical vectors
from sklearn.feature_extraction.text import CountVectorizer
# Initialize CountVectorizer keeping top 5000 most frequent words and removing common English stop words (is, the, and)
cv = CountVectorizer(max_features=5000, stop_words='english')
# Fit and transform text tags into a 2D numpy array of word count vectors
vectors = cv.fit_transform(new_df['tags']).toarray()

# Import cosine_similarity to measure the angle/similarity between movie vector representations
from sklearn.metrics.pairwise import cosine_similarity
# Compute cosine similarity matrix (compares every movie with every other movie)
similarity = cosine_similarity(vectors)
# Print matrix dimensions (number of movies x number of movies)
print(f"Similarity Matrix Shape: {similarity.shape}")

Similarity Matrix Shape: (4806, 4806)


In [13]:
# Define the recommendation function that accepts a movie name and outputs top 5 similar movies
def recommend(movie):
    # Find matching row in DataFrame regardless of letter casing
    matches = new_df[new_df['title'].str.lower() == movie.lower()]
    # Check if movie exists in the dataset
    if matches.empty:
        # Inform user if movie is not found
        print(f"Movie '{movie}' not found!")
        return
    # Fetch the index of the queried movie
    movie_index = matches.index[0]
    # Retrieve similarity scores of this movie with all other movies
    distances = similarity[movie_index]
    # Sort movies by similarity score in descending order and slice top 5 (ignoring index 0 which is the movie itself)
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    # Print header for recommendations
    print(f"Recommendations for '{new_df.iloc[movie_index].title}':\n")
    # Loop through recommended movie indices and display their titles
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [14]:
# Test the recommendation function with 'Avatar' to verify that similar sci-fi / action movies are returned
recommend('Avatar')

Recommendations for 'Avatar':

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [15]:
# Import pickle module to serialize and save Python objects to binary files
import pickle
# Save the clean movies DataFrame as a dictionary for building web apps (Streamlit / Flask)
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))
# Save the computed similarity matrix to disk so we don't have to recalculate it every time
pickle.dump(similarity, open('similarity.pkl', 'wb'))
# Print confirmation message
print("Model files saved: movie_dict.pkl and similarity.pkl")

Model files saved: movie_dict.pkl and similarity.pkl
